In [ ]:
# STEP 1: Mount Drive (Optional, for saving models)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# STEP 2: Setup Kaggle API and Download CADI-AI Dataset
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()  # Upload your kaggle.json here

# Configure kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip the CADI-AI dataset
!kaggle datasets download -d karaagroaiprojects/cadi-ai -p /content/Data --unzip


In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm


In [ ]:
import torch
import torch.nn as nn

# SE block (channel-wise attention)
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.view(b, c, -1).mean(dim=2)
        y = torch.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y))
        y = y.view(b, c, 1, 1)
        return x * y.expand_as(x)

# Ghost Module
class GhostModule(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.primary = nn.Conv2d(in_channels, out_channels // 2, 1)
        self.ghost = nn.Conv2d(out_channels // 2, out_channels // 2, 3, padding=1, groups=out_channels // 2)

    def forward(self, x):
        x1 = self.primary(x)
        x2 = self.ghost(x1)
        return torch.cat([x1, x2], dim=1)

# Depthwise Separable Conv
class DSConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, padding=1, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

# Eff-UAVNet
class EffUAVNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.block1 = DSConv(16, 32)
        self.block2 = GhostModule(32, 64)
        self.se = SEBlock(64)
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.initial(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.se(x)
        x = self.block3(x)
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc(x)


In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import random_split, DataLoader

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder('/content/Data/cadi-ai', transform=transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)

num_classes = len(dataset.classes)
print(f"Classes: {dataset.classes}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EffUAVNet(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train the model
for epoch in range(10):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/10], Loss: {total_loss:.4f}")


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")
